## Vehicle Insurance Claim Fraud Detection

In [253]:
import kagglehub
import pandas as pd
import numpy as np

## Load the dataset

In [219]:
path = kagglehub.dataset_download("shivamb/vehicle-claim-fraud-detection")
df = pd.read_csv(f"{path}/fraud_oracle.csv")
df.head()

,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


In [220]:
df.shape

(15420, 33)

In [221]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Month                 15420 non-null  str  
 1   WeekOfMonth           15420 non-null  int64
 2   DayOfWeek             15420 non-null  str  
 3   Make                  15420 non-null  str  
 4   AccidentArea          15420 non-null  str  
 5   DayOfWeekClaimed      15420 non-null  str  
 6   MonthClaimed          15420 non-null  str  
 7   WeekOfMonthClaimed    15420 non-null  int64
 8   Sex                   15420 non-null  str  
 9   MaritalStatus         15420 non-null  str  
 10  Age                   15420 non-null  int64
 11  Fault                 15420 non-null  str  
 12  PolicyType            15420 non-null  str  
 13  VehicleCategory       15420 non-null  str  
 14  VehiclePrice          15420 non-null  str  
 15  FraudFound_P          15420 non-null  int64
 16  PolicyNumber   

In [222]:
df.isna().sum()

Month                   0
WeekOfMonth             0
DayOfWeek               0
Make                    0
AccidentArea            0
DayOfWeekClaimed        0
MonthClaimed            0
WeekOfMonthClaimed      0
Sex                     0
MaritalStatus           0
Age                     0
Fault                   0
PolicyType              0
VehicleCategory         0
VehiclePrice            0
FraudFound_P            0
PolicyNumber            0
RepNumber               0
Deductible              0
DriverRating            0
Days_Policy_Accident    0
Days_Policy_Claim       0
PastNumberOfClaims      0
AgeOfVehicle            0
AgeOfPolicyHolder       0
PoliceReportFiled       0
WitnessPresent          0
AgentType               0
NumberOfSuppliments     0
AddressChange_Claim     0
NumberOfCars            0
Year                    0
BasePolicy              0
dtype: int64

In [223]:
#target class is highly imbalanced
df['FraudFound_P'].value_counts() / df.shape[0] * 100

FraudFound_P
0    94.014267
1     5.985733
Name: count, dtype: float64

In [224]:
columns_to_check = ['Days_Policy_Accident', 'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'NumberOfSuppliments', 'NumberOfCars']

for col in columns_to_check:
    print(f"{col}: {df[col].unique()}")

Days_Policy_Accident: <StringArray>
['more than 30', '15 to 30', 'none', '1 to 7', '8 to 15']
Length: 5, dtype: str
PastNumberOfClaims: <StringArray>
['none', '1', '2 to 4', 'more than 4']
Length: 4, dtype: str
AgeOfVehicle: <StringArray>
[    '3 years',     '6 years',     '7 years', 'more than 7',     '5 years',
         'new',     '4 years',     '2 years']
Length: 8, dtype: str
AgeOfPolicyHolder: <StringArray>
['26 to 30', '31 to 35', '41 to 50', '51 to 65', '21 to 25', '36 to 40',
 '16 to 17',  'over 65', '18 to 20']
Length: 9, dtype: str
NumberOfSuppliments: <StringArray>
['none', 'more than 5', '3 to 5', '1 to 2']
Length: 4, dtype: str
NumberOfCars: <StringArray>
['3 to 4', '1 vehicle', '2 vehicles', '5 to 8', 'more than 8']
Length: 5, dtype: str


In [225]:
df.isnull().sum()

Month                   0
WeekOfMonth             0
DayOfWeek               0
Make                    0
AccidentArea            0
DayOfWeekClaimed        0
MonthClaimed            0
WeekOfMonthClaimed      0
Sex                     0
MaritalStatus           0
Age                     0
Fault                   0
PolicyType              0
VehicleCategory         0
VehiclePrice            0
FraudFound_P            0
PolicyNumber            0
RepNumber               0
Deductible              0
DriverRating            0
Days_Policy_Accident    0
Days_Policy_Claim       0
PastNumberOfClaims      0
AgeOfVehicle            0
AgeOfPolicyHolder       0
PoliceReportFiled       0
WitnessPresent          0
AgentType               0
NumberOfSuppliments     0
AddressChange_Claim     0
NumberOfCars            0
Year                    0
BasePolicy              0
dtype: int64

In [226]:
days_policy_accident_map = {
    'none': 0,
    '1 to 7': 1,
    '8 to 15': 2,
    '15 to 30': 3,
    'more than 30': 4
}

df['Days_Policy_Accident'] = df['Days_Policy_Accident'].map(days_policy_accident_map)

In [227]:
days_past_number_map = {
    'none': 0,
    '1': 1,
    '2 to 4': 2,
    'more than 4': 3
}

df['PastNumberOfClaims'] = df['PastNumberOfClaims'].map(days_past_number_map)

In [228]:
days_age_vehicle_map = {
    'new': 0,
    '2 years': 1,
    '3 years': 2,
    '4 years': 3,
    '5 years': 4,
    '6 years': 5,
    '7 years': 6,
    'more than 7': 7
}

df['AgeOfVehicle'] = df['AgeOfVehicle'].map(days_age_vehicle_map)

In [229]:
days_age_policy_holder_map = {
    '16 to 17': 0,
    '18 to 20': 1,
    '21 to 25': 2,
    '26 to 30': 3,
    '31 to 35': 4,
    '36 to 40': 5,
    '41 to 50': 6,
    '51 to 65': 7,
    'over 65': 8
}

df['AgeOfPolicyHolder'] = df['AgeOfPolicyHolder'].map(days_age_policy_holder_map)

In [230]:
days_number_supplements_map = {
    'none': 0,
    '1 to 2': 1,
    '3 to 5': 2,
    'more than 5': 3
}

df['NumberOfSuppliments'] = df['NumberOfSuppliments'].map(days_number_supplements_map)

In [231]:
days_number_cars_map = {
    '1 vehicle': 0,
    '2 vehicles': 1,
    '3 to 4': 2,
    '5 to 8': 3,
    'more than 8': 4
}

df['NumberOfCars'] = df['NumberOfCars'].map(days_number_cars_map)

In [232]:
df['VehiclePrice'].unique()

<StringArray>
['more than 69000',  '20000 to 29000',  '30000 to 39000', 'less than 20000',
  '40000 to 59000',  '60000 to 69000']
Length: 6, dtype: str

In [233]:
days_vehicle_price_map = {
    'less than 20000': 0,
    '20000 to 29000': 1,
    '30000 to 39000': 2,
    '40000 to 59000': 3,
    '60000 to 69000': 4,
    'more than 69000': 4
}

df['VehiclePrice'] = df['VehiclePrice'].map(days_vehicle_price_map)

<StringArray>
['more than 30', '15 to 30', '8 to 15', 'none']
Length: 4, dtype: str

In [235]:
days_days_policy_map = {
    'none': 0,
    '8 to 15': 1,
    '15 to 30': 2,
    'more than 30': 3
}

df['Days_Policy_Claim'] = df['Days_Policy_Claim'].map(days_days_policy_map)

In [236]:
df['Days_Policy_Claim'].unique()

array([3, 2, 1, 0])

In [238]:
print(df['PolicyNumber'].nunique() == len(df))

True


In [239]:
df = df.drop(columns=['PolicyNumber'])

In [240]:
df['MonthClaimed'].isna().sum()

np.int64(0)

In [241]:
print((df['MonthClaimed'] == '0').sum())
print(df[df['MonthClaimed'] == '0']['FraudFound_P'].mean())

1
0.0


In [242]:
df = df[df['MonthClaimed'] != '0']

In [243]:
df['WeekOfMonth'].unique()

array([5, 3, 2, 4, 1])

In [244]:
df['WeekOfMonthClaimed'].unique()

array([1, 4, 2, 3, 5])

In [245]:
month_mapping = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12
}
df['Month'] = df['Month'].map(month_mapping)
df['MonthClaimed'] = df['MonthClaimed'].map(month_mapping)
df['MonthClaimed'] = df['MonthClaimed'].astype(int)

In [246]:
month_diff = df['MonthClaimed'] - df['Month']
month_diff = month_diff.where(month_diff >= 0, month_diff + 12)

df['claim_delay_approx'] = month_diff * 4 + (df['WeekOfMonthClaimed'] - df['WeekOfMonth'])

In [247]:
df['claim_delay_approx'].describe()

count    15419.000000
mean         1.405409
std          3.403515
min         -3.000000
25%          0.000000
50%          1.000000
75%          1.000000
max         48.000000
Name: claim_delay_approx, dtype: float64

In [248]:
df[df['claim_delay_approx'] < 0][['Month', 'WeekOfMonth', 'MonthClaimed', 'WeekOfMonthClaimed', 'claim_delay_approx']]

,Month,WeekOfMonth,MonthClaimed,WeekOfMonthClaimed,claim_delay_approx
5247,9,4,9,3,-1
9553,4,3,4,1,-2
14339,11,5,11,2,-3


In [249]:
#we can do that cause there are only 3 rows
df['claim_delay_approx'] = df['claim_delay_approx'].clip(lower=0)

In [250]:
categorical_cols = ['Make', 'AccidentArea', 'Sex', 'MaritalStatus', 'Fault',
                     'PolicyType', 'VehicleCategory', 'PoliceReportFiled',
                     'WitnessPresent', 'AgentType', 'AddressChange_Claim',
                     'BasePolicy', 'DayOfWeek', 'DayOfWeekClaimed']

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [251]:
df.head()

,Month,WeekOfMonth,MonthClaimed,WeekOfMonthClaimed,Age,VehiclePrice,FraudFound_P,RepNumber,Deductible,DriverRating,...,DayOfWeek_Sunday,DayOfWeek_Thursday,DayOfWeek_Tuesday,DayOfWeek_Wednesday,DayOfWeekClaimed_Monday,DayOfWeekClaimed_Saturday,DayOfWeekClaimed_Sunday,DayOfWeekClaimed_Thursday,DayOfWeekClaimed_Tuesday,DayOfWeekClaimed_Wednesday
0,12,5,1,1,21,4,0,12,300,1,...,False,False,False,True,False,False,False,False,True,False
1,1,3,1,4,34,4,0,15,400,4,...,False,False,False,True,True,False,False,False,False,False
2,10,5,11,2,47,4,0,7,400,3,...,False,False,False,False,False,False,False,True,False,False
3,6,2,7,1,65,1,0,4,400,2,...,False,False,False,False,False,False,False,False,False,False
4,1,5,2,2,27,4,0,3,400,1,...,False,False,False,False,False,False,False,False,True,False


In [252]:
print(df.shape)
print(df.dtypes.value_counts())

(15419, 74)
bool     55
int64    19
Name: count, dtype: int64


In [255]:
## adding some null values to the dataset for my training purpose
np.random.seed(42)

for col in ['Age', 'Deductible', 'DriverRating']:
    mask = np.random.rand(len(df)) < 0.1  # 10% missing values
    df.loc[mask, col] = np.nan